In [52]:
from langgraph.graph import StateGraph, START, END
from langgraph.types import interrupt, Command
from langgraph.checkpoint.memory import InMemorySaver

from typing import TypedDict

In [53]:
from typing import Literal, Union


class State(TypedDict):
    status: Literal["approved", "rejected"]


In [54]:
def get_approval(state):
    approval = interrupt(value="please approve or reject")

    if approval == "approve":
        return {"status" : "approved"}
    return {"status" : "rejected"}
    
def accepted(state):
    print("accepted")
def rejected(state):
    pass

def route(state):
    status = state["status"]

    if status == "approved":
        return "approved"
 
    return "rejected"

In [55]:
checkpointer = InMemorySaver()

graph = StateGraph(State)

graph.add_node("get_approval", get_approval)
graph.add_node("accepted", accepted)
graph.add_node("rejected", rejected)

graph.add_edge(START, "get_approval")
graph.add_conditional_edges("get_approval", route , {
    "approved" : "accepted",
    "rejected" : "rejected"
})

graph.add_edge("accepted", END)
graph.add_edge("rejected", END)


In [56]:


config = {
    "configurable": {
        "thread_id": "prep-001"
    }
}


workflow = graph.compile(checkpointer=checkpointer)
workflow.invoke({}, config=config)

approval = input("Please approve/reject: ")

workflow.invoke(Command(resume = approval), config=config)

accepted


{'status': 'approved'}